# Test Set Evaluation + Detection Samples

Runs test-split evaluation on all 12 trained models and generates sample detection images.
Takes ~15 minutes (no training, just inference).

In [ ]:
!pip install ultralytics --quiet

In [ ]:
import os
import json
import numpy as np
import pandas as pd
from ultralytics import YOLO

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# =============================================================================
# PATHS
# =============================================================================
OUTPUT_ROOT = "/content/drive/MyDrive/ForensicsDataset/rerun_experiments_v2"
DATA_YAML = "/content/drive/MyDrive/ForensicsDataset/merged_dataset/data.yaml"
TEST_IMAGES = "/content/drive/MyDrive/ForensicsDataset/merged_dataset/images/test"

MODELS = ["yolov8m_baseline", "yolov8m_enhanced", "yolo11m_baseline", "yolo11m_enhanced"]
SEEDS = [42, 123, 7]

## 1. Verify data.yaml has test path

In [ ]:
import yaml

with open(DATA_YAML, "r") as f:
    data_cfg = yaml.safe_load(f)

print("Dataset config:")
for k, v in data_cfg.items():
    if k != "names":
        print(f"  {k}: {v}")

# Check test path exists
test_path = data_cfg.get("test", None)
if test_path and os.path.exists(test_path):
    n_test = len([f for f in os.listdir(test_path) if f.endswith(('.jpg','.png','.jpeg'))])
    print(f"\n[OK] Test path exists: {test_path}")
    print(f"[OK] Test images found: {n_test}")
else:
    print(f"\n[WARNING] Test path not found or not set: {test_path}")
    print("You may need to add 'test: path/to/test/images' to data.yaml")

## 2. Run test set evaluation on all models

In [ ]:
results_all = []

for model_name in MODELS:
    for seed in SEEDS:
        weights = os.path.join(
            OUTPUT_ROOT, model_name, f"seed_{seed}", "train", "weights", "best.pt"
        )
        if not os.path.exists(weights):
            print(f"[SKIP] {model_name} seed={seed} — weights not found")
            continue

        print(f"\n{'='*60}")
        print(f"TEST EVAL: {model_name} | seed={seed}")
        print(f"{'='*60}")

        model = YOLO(weights)
        results = model.val(
            data=DATA_YAML,
            split="test",
            batch=8,
            imgsz=640,
            device=0,
            plots=True,
            save_json=True,
            project=os.path.join(OUTPUT_ROOT, model_name, f"seed_{seed}"),
            name="test_eval",
            exist_ok=True,
        )

        # Confirm image count
        n_images = results.box.n if hasattr(results.box, 'n') else "unknown"
        print(f"  Images evaluated: {n_images}")
        print(f"  Precision: {results.box.mp:.4f}")
        print(f"  Recall:    {results.box.mr:.4f}")
        print(f"  mAP@50:    {results.box.map50:.4f}")
        print(f"  mAP@50-95: {results.box.map:.4f}")

        # Per-class AP
        per_class_ap50 = {}
        try:
            class_names = list(results.names.values())
            ap50_vals = results.box.ap50.tolist()
            per_class_ap50 = dict(zip(class_names, ap50_vals))
        except Exception:
            pass

        # Domain split: crime scene = classes 0-37, smart device = classes 38-47
        cs_aps = [v for i, (k, v) in enumerate(per_class_ap50.items()) if i < 38]
        sd_aps = [v for i, (k, v) in enumerate(per_class_ap50.items()) if i >= 38]

        results_all.append({
            "model": model_name,
            "seed": seed,
            "n_images": int(n_images) if isinstance(n_images, (int, float)) else None,
            "precision": float(results.box.mp),
            "recall": float(results.box.mr),
            "mAP50": float(results.box.map50),
            "mAP50_95": float(results.box.map),
            "mAP50_crime_scene": float(np.mean(cs_aps)) if cs_aps else None,
            "mAP50_smart_device": float(np.mean(sd_aps)) if sd_aps else None,
        })

# Save raw results
df = pd.DataFrame(results_all)
df.to_csv(os.path.join(OUTPUT_ROOT, "test_set_results.csv"), index=False)
print(f"\nSaved: {OUTPUT_ROOT}/test_set_results.csv")
print(df.to_string(index=False))

## 3. Summary table (mean ± std)

In [ ]:
summary = df.groupby("model").agg(
    n_images=("n_images", "first"),
    P_mean=("precision", "mean"), P_std=("precision", "std"),
    R_mean=("recall", "mean"), R_std=("recall", "std"),
    mAP50_mean=("mAP50", "mean"), mAP50_std=("mAP50", "std"),
    mAP95_mean=("mAP50_95", "mean"), mAP95_std=("mAP50_95", "std"),
    cs_mean=("mAP50_crime_scene", "mean"), cs_std=("mAP50_crime_scene", "std"),
    sd_mean=("mAP50_smart_device", "mean"), sd_std=("mAP50_smart_device", "std"),
).round(4)

print("=" * 70)
print("TEST SET RESULTS (mean +/- std across 3 seeds)")
print("=" * 70)
print(summary.to_string())

summary.to_csv(os.path.join(OUTPUT_ROOT, "test_set_summary.csv"))
print(f"\nSaved: {OUTPUT_ROOT}/test_set_summary.csv")

## 4. Generate detection sample images
Runs the best model (YOLO11-m baseline, seed=42) on test images and saves annotated results.

In [ ]:
# Load best model
best_weights = os.path.join(
    OUTPUT_ROOT, "yolo11m_baseline", "seed_42", "train", "weights", "best.pt"
)
assert os.path.exists(best_weights), f"Best weights not found: {best_weights}"
model = YOLO(best_weights)

# Run prediction on test images
save_dir = os.path.join(OUTPUT_ROOT, "detection_samples")
results = model.predict(
    source=TEST_IMAGES,
    save=True,
    conf=0.5,
    imgsz=640,
    project=save_dir,
    name="yolo11m_best",
    exist_ok=True,
    max_det=50,
)

print(f"\nDetection samples saved to: {save_dir}/yolo11m_best/")
print(f"Total images processed: {len(results)}")
print("\nBrowse the folder and pick 4 images for the paper:")
print("  - A cluttered crime scene with multiple detections")
print("  - A clean smart-device image")
print("  - A challenging success case (small/occluded evidence)")
print("  - A failure case (missed or misclassified object)")

## 5. Also run YOLOv8-m baseline for comparison samples

In [ ]:
v8_weights = os.path.join(
    OUTPUT_ROOT, "yolov8m_baseline", "seed_42", "train", "weights", "best.pt"
)
if os.path.exists(v8_weights):
    model_v8 = YOLO(v8_weights)
    results_v8 = model_v8.predict(
        source=TEST_IMAGES,
        save=True,
        conf=0.5,
        imgsz=640,
        project=save_dir,
        name="yolov8m_best",
        exist_ok=True,
        max_det=50,
    )
    print(f"YOLOv8-m samples saved to: {save_dir}/yolov8m_best/")
else:
    print("[SKIP] YOLOv8-m weights not found")

## 6. Done

In [ ]:
print(f"""
{'='*70}
COMPLETE
{'='*70}
Test evaluation results:
  {OUTPUT_ROOT}/test_set_results.csv     (per-run)
  {OUTPUT_ROOT}/test_set_summary.csv     (mean +/- std)

Detection samples for paper figures:
  {save_dir}/yolo11m_best/    (best model)
  {save_dir}/yolov8m_best/    (comparison)

Next: pick 4 images for the 2x2 figure and send me
the test_set_results.csv to update the paper tables.
{'='*70}
""")